# Heat Equation Solver: FTCS vs BTCS
This notebook demonstrates the numerical solution of the 1D Heat Equation:
$$\frac{\partial u}{\partial t} = \alpha \frac{\partial^2 u}{\partial x^2}$$
using Forward Time Centered Space (Explicit) and Backward Time Centered Space (Implicit) methods.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
from scipy.sparse import diags
from scipy.sparse.linalg import spsolve

# Parameters
L = 1.0        # Length of the domain
Nx = 50        # Number of spatial points
dx = L / (Nx - 1)
x = np.linspace(0, L, Nx)

alpha = 0.01   # Thermal diffusivity
T_total = 2.0  # Total time
dt = 0.01      # Time step
Nt = int(T_total / dt)

# Stability check for FTCS
r = alpha * dt / dx**2
print(f"Stability parameter r: {r:.4f}")
if r > 0.5:
    print("Warning: FTCS will be unstable (r > 0.5)")

# Initial condition: A pulse in the middle
u0 = np.exp(-100 * (x - 0.5)**2)

def solve_ftcs(u0, Nt, r):
    u = u0.copy()
    history = [u.copy()]
    for _ in range(Nt):
        u[1:-1] = u[1:-1] + r * (u[2:] - 2*u[1:-1] + u[0:-2])
        history.append(u.copy())
    return np.array(history)

def solve_btcs(u0, Nt, r):
    u = u0.copy()
    history = [u.copy()]
    # Matrix for implicit step: (1 + 2r)u_i - r*u_{i-1} - r*u_{i+1} = u_old
    main_diag = (1 + 2*r) * np.ones(Nx - 2)
    off_diag = -r * np.ones(Nx - 3)
    A = diags([off_diag, main_diag, off_diag], [-1, 0, 1]).tocsr()

    for _ in range(Nt):
        b = u[1:-1]
        u[1:-1] = spsolve(A, b)
        history.append(u.copy())
    return np.array(history)

# Run solvers
hist_ftcs = solve_ftcs(u0, Nt, r)
hist_btcs = solve_btcs(u0, Nt, r)

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# Animation Setup
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
line1, = ax1.plot(x, u0, 'r-', label='FTCS (Explicit)')
line2, = ax2.plot(x, u0, 'b-', label='BTCS (Implicit)')

for ax in [ax1, ax2]:
    ax.set_ylim(-0.1, 1.1)
    ax.set_xlabel('x')
    ax.set_ylabel('Temperature')
    ax.legend()

def update(frame):
    line1.set_ydata(hist_ftcs[frame])
    line2.set_ydata(hist_btcs[frame])
    ax1.set_title(f"Time: {frame*dt:.2f}s")
    return line1, line2

ani = FuncAnimation(fig, update, frames=range(0, Nt, 2), interval=50, blit=True)
plt.close()
HTML(ani.to_jshtml())